# Gemini Token Optimization: Apple-to-Apple Comparison

This notebook compares the same support-policy task before and after several Gemini token optimization approaches. The comparison is apple-to-apple: the user question and required answer remain stable, while only the context packaging or execution strategy changes.

The notebook is configured for live Gemini calls only: use `gcloud auth application-default login`, then run the cells to call Gemini `count_tokens` and `generate_content`.


## What Gets Measured

- Input tokens: counted before sending a request with Gemini `count_tokens`.
- Output tokens: measured from real Gemini `generate_content` responses through `response.usage_metadata`.
- Cost: `(normal_input_tokens * input_price) + (cached_input_tokens * cached_price) + (output_tokens * output_price) + cache_storage`.
- Saving percent: `(before_cost - after_cost) / before_cost * 100`.

Important nuance: context caching may not reduce the logical prompt token count, but it can reduce cost because cached input tokens are billed differently from normal input tokens. The `Local NLP first` row intentionally skips Gemini for the optimized path, so that row has a local answer by design.


## Official Docs Used

- ADK Python quickstart: https://adk.dev/get-started/python/
- ADK Gemini model auth: https://adk.dev/agents/models/google-gemini/
- ADK context caching: https://adk.dev/context/caching/
- ADK context compaction: https://adk.dev/context/compaction/
- Gemini token counting: https://ai.google.dev/gemini-api/docs/tokens
- Gemini pricing: https://ai.google.dev/gemini-api/docs/pricing


In [1]:
# Install missing notebook-kernel dependencies before importing the project modules.
# This uses the same Python executable as the active Jupyter kernel.
import importlib
import subprocess
import sys
from pathlib import Path

required_modules = {
    "google.genai": "google-genai",
    "google.adk": "google-adk",
    "pandas": "pandas",
}

missing = []
for module_name, package_name in required_modules.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing.append(package_name)

if missing:
    requirements_path = Path("requirements.txt")
    if requirements_path.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *sorted(set(missing))])
    importlib.invalidate_caches()

print("Notebook dependencies are ready.")
# If needed, authenticate once from a terminal or uncomment the next line:
# !gcloud auth application-default login


  Using cached aiosqlite-0.22.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached authlib-1.7.2-py2.py3-none-any.whl.metadata (10 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyasn1-0.6.3-py3-none-any.whl.metadata (8.4 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 6.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 6.4 MB/s  0:00:00
Using cached authlib-1.7.2-py2.py3-none-any.whl (259 kB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached aiosqlite-0.22.1-py3-none-any.whl (17 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 8.0 MB/s  0:00:00 eta 0:00:01
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
Using cached pyasn1-0.6.3-py3-none-any.whl (83 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 6.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.0 MB/s  0:00:00 eta 0:00:01
  Created wheel for watchdog: filename=watchdog-6.0.0-cp314-cp314-macosx_12_0_arm64.whl size=87914 sha256=bf29dfdc85857b6fb49dceae14ef3f68701808fc1157fdb92fbfad98c1e78dde
  Stored in directory: /Users/ari

In [2]:
from config import DEFAULT_PRICES, apply_environment, create_genai_client, get_active_gcloud_project, load_config
from token_tools import (
    build_sample_dataset,
    calculate_cost,
    compact_history,
    compare_cached_reuse,
    compare_prompts,
    count_tokens,
    format_history,
    generate_and_measure,
    local_nlp_answer,
    make_agent_prompt,
    select_relevant_sections,
)
import pandas as pd

# Use the active gcloud project by default.
PROJECT_ID = get_active_gcloud_project()
LOCATION = "global"  # use your supported Agent Platform / Vertex location, for example "global" or "us-central1"
MODEL = "gemini-2.5-flash"

# Live-only mode: use Gemini for token counting and generated answers.
USE_LIVE_GEMINI = True

# This spends tokens with generate_content so output answers and usage metadata are real.
RUN_LIVE_GENERATION = True
STRICT_GEMINI_COUNTING = True

cfg = load_config(project=PROJECT_ID or None, location=LOCATION, model=MODEL)
apply_environment(cfg)

if not USE_LIVE_GEMINI or not RUN_LIVE_GENERATION:
    raise ValueError("This notebook is configured for live Gemini only. Keep USE_LIVE_GEMINI and RUN_LIVE_GENERATION set to True.")
if cfg.use_enterprise and not cfg.project:
    raise ValueError("No active project found. Run `gcloud config set project YOUR_PROJECT_ID` first.")
client = create_genai_client(cfg)

print(f"Model: {cfg.model}")
print(f"Project: {cfg.project}")
print(f"Location: {cfg.location}")
print("Counting mode: Gemini count_tokens (strict)")
print("Pricing:", cfg.pricing)


Model: gemini-2.5-flash
Project: appl-int-df-prd-wd2y
Location: global
Counting mode: Gemini count_tokens (strict)
Pricing: ModelPricing(input_per_million=0.3, output_per_million=2.5, cached_input_per_million=0.075, cache_storage_per_million_token_hour=1.0)


## Sample Task

The sample uses one realistic support-policy question and a mixed context pack: policy text, operational notes, regional notes, and previous chat turns. Each optimization below answers the same question.


In [3]:
sample = build_sample_dataset()
question = sample["question"]
full_context = sample["full_context"]
policy_document = sample["policy_document"]
full_history = format_history(sample["history"])

print("Question:")
print(question)
print("\nFull Context:")
print(full_context)
print("\nContext characters:", len(full_context))
print("\nHistory:")
print(sample["history"])
print("\nHistory turns:", len(sample["history"]))
print("Gemini-counted full context tokens:", count_tokens(full_context, model=cfg.model, client=client, allow_estimate_fallback=not STRICT_GEMINI_COUNTING).tokens)


Question:
A customer in Jakarta bought an annual Pro self-serve plan 12 days ago and wants a refund. What should support answer and what internal steps should be followed?

Full Context:
## Refund policy
Customers on monthly or annual self-serve plans have a 30-day refund window from the original purchase date. Support can approve one courtesy refund per account per calendar year. Refunds are blocked when there is evidence of account abuse, chargeback fraud, or more than 10,000 successful API calls after the purchase. Annual plan refunds should be prorated only after the 30-day window; inside the window, process a full refund. Before processing, verify the account email, workspace ID, payment processor ID, and invoice number. After processing, send the approved refund template and tag the ticket with billing_refund.

## Enterprise billing policy
Enterprise contracts are handled by Finance Ops and must not be refunded directly by support. Contract cancellation requests require the accou

/Users/aries_fitriawan/anaconda3/envs/pycon26/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Gemini-counted full context tokens: 650


## Baseline Prompt

Baseline behavior sends all available context and the full conversation history. This is intentionally wasteful so the savings from each pattern are visible.


In [4]:
BASELINE_OUTPUT_TOKENS = 1024
CONCISE_OUTPUT_TOKENS = 512
JSON_OUTPUT_TOKENS = 512

#Newbie Prompt
baseline_contract = (
    "Write a detailed customer-facing response and include internal support steps. "
    "Mention policy reasoning, regional caveats, and any verification needed."
)

#Consice Prompt
concise_contract = (
    "Answer in at most 5 bullets. Include eligibility, customer reply, internal steps, "
    "and one caveat only if relevant."
)

#JSON Prompt
json_contract = (
    "Return compact JSON with keys: eligible, customer_reply, internal_steps, caveat. "
    "Do not include background text."
)

baseline_prompt = make_agent_prompt(
    question=question,
    context=full_context,
    history=full_history,
    output_contract=baseline_contract,
)

concise_prompt = make_agent_prompt(
    question=question,
    context=full_context,
    history=full_history,
    output_contract=concise_contract,
)

json_prompt = make_agent_prompt(
    question=question,
    context=full_context,
    history=full_history,
    output_contract=json_contract,
)

baseline_count = count_tokens(baseline_prompt, model=cfg.model, client=client, allow_estimate_fallback=not STRICT_GEMINI_COUNTING)
print("Baseline input tokens:", baseline_count.tokens, f"({baseline_count.method})")
print("Baseline max output token cap:", BASELINE_OUTPUT_TOKENS)

concise_count = count_tokens(concise_prompt, model=cfg.model, client=client, allow_estimate_fallback=not STRICT_GEMINI_COUNTING)
print("\nConcise input tokens:", concise_count.tokens, f"({concise_count.method})")
print("Concise max output token cap:", CONCISE_OUTPUT_TOKENS)

json_count = count_tokens(json_prompt, model=cfg.model, client=client, allow_estimate_fallback=not STRICT_GEMINI_COUNTING)
print("\nJSON input tokens:", json_count.tokens, f"({json_count.method})")
print("JSON max output token cap:", JSON_OUTPUT_TOKENS)


Baseline input tokens: 914 (gemini)
Baseline max output token cap: 1024

Concise input tokens: 913 (gemini)
Concise max output token cap: 512

JSON input tokens: 913 (gemini)
JSON max output token cap: 512


## Build Before/After Rows

Each row compares the baseline-style request against one optimized variant. Input tokens are counted with Gemini `count_tokens`; output tokens are measured from real Gemini `generate_content` usage metadata. Max output caps are held equal when the approach should affect only input context, and reduced only for approaches that explicitly constrain output.


In [5]:
rows = []

generation_cache = {}

def get_live_answer(prompt, *, model, max_output_tokens):
    """Generate once per unique prompt/model/output cap and reuse the result."""
    cache_key = (model, max_output_tokens, prompt)
    if cache_key not in generation_cache:
        generation_cache[cache_key] = generate_and_measure(
            prompt,
            model=model,
            client=client,
            max_output_tokens=max_output_tokens,
            temperature=0,
        )
    return generation_cache[cache_key]

def output_tokens_from_usage(usage):
    if usage.get("candidates_token_count") is not None:
        return usage["candidates_token_count"]
    if usage.get("total_token_count") is not None and usage.get("prompt_token_count") is not None:
        return max(usage["total_token_count"] - usage["prompt_token_count"], 0)
    raise ValueError(f"Cannot read output token count from usage metadata: {usage}")

def pricing_for_model(model_name):
    if model_name == cfg.model:
        return cfg.pricing
    return DEFAULT_PRICES.get(model_name, cfg.pricing)

def saving_pct(before, after):
    return 0.0 if before <= 0 else (before - after) / before * 100

def attach_answer(row, *, before_prompt, before_max_output_tokens, prompt, max_output_tokens, model=None, local_answer_text=None, repeat_calls=1):
    """Attach live before/after answers and recompute cost with real output tokens."""
    row = dict(row)
    before_model = row["before_model"]
    after_model = model or row["after_model"]

    before_answer, before_usage = get_live_answer(
        before_prompt,
        model=before_model,
        max_output_tokens=before_max_output_tokens,
    )
    before_output_tokens = output_tokens_from_usage(before_usage) * repeat_calls

    if local_answer_text is not None and not prompt:
        after_answer = local_answer_text
        after_usage = {}
        after_output_tokens = 0
        answer_source = "local NLP (Gemini skipped by optimization)"
    else:
        after_answer, after_usage = get_live_answer(
            prompt,
            model=after_model,
            max_output_tokens=max_output_tokens,
        )
        after_output_tokens = output_tokens_from_usage(after_usage) * repeat_calls
        answer_source = "live Gemini"

    before_cost = calculate_cost(
        input_tokens=row["before_input_tokens"],
        output_tokens=before_output_tokens,
        pricing=pricing_for_model(before_model),
        cached_input_tokens=row.get("before_cached_input_tokens", 0),
        cache_storage_token_hours=row.get("before_cache_storage_token_hours", 0),
    )
    after_cost = calculate_cost(
        input_tokens=row["after_input_tokens"],
        output_tokens=after_output_tokens,
        pricing=pricing_for_model(after_model),
        cached_input_tokens=row.get("after_cached_input_tokens", 0),
        cache_storage_token_hours=row.get("after_cache_storage_token_hours", 0),
    )

    row["before_answer"] = before_answer.strip()
    row["optimized_answer"] = after_answer.strip()
    row["answer_source"] = answer_source
    row["before_usage_metadata"] = before_usage
    row["answer_usage_metadata"] = after_usage
    row["before_output_tokens"] = before_output_tokens
    row["after_output_tokens"] = after_output_tokens
    row["output_token_saving_pct"] = saving_pct(before_output_tokens, after_output_tokens)
    row["before_cost_usd"] = before_cost.total_usd
    row["after_cost_usd"] = after_cost.total_usd
    row["cost_saving_pct"] = saving_pct(before_cost.total_usd, after_cost.total_usd)
    return row

# 1. Lazy loading / retrieval: send only sections relevant to the same question, especially on RAG.
lazy_context = select_relevant_sections(full_context, question, max_sections=3)
lazy_prompt_same_output = make_agent_prompt(
    question=question,
    context=lazy_context,
    history=full_history,
    output_contract=baseline_contract,
)

rows.append(
    attach_answer(
        compare_prompts(
            approach="Lazy loading context",
            before_prompt=baseline_prompt,
            after_prompt=lazy_prompt_same_output,
            model=cfg.model,
            before_pricing=cfg.pricing,
            client=client,
            before_output_tokens=BASELINE_OUTPUT_TOKENS,
            after_output_tokens=BASELINE_OUTPUT_TOKENS,
            allow_estimate_fallback=not STRICT_GEMINI_COUNTING,
            note="Same answer budget; only irrelevant context is removed.",
        ),
        before_prompt=baseline_prompt,
        before_max_output_tokens=BASELINE_OUTPUT_TOKENS,
        prompt=lazy_prompt_same_output,
        max_output_tokens=BASELINE_OUTPUT_TOKENS,
    )
)

# 2. Local NLP first: deterministic policy extraction answers without Gemini.
local_answer = local_nlp_answer(question, policy_document)
rows.append(
    attach_answer(
        compare_prompts(
            approach="Local NLP first",
            before_prompt=baseline_prompt,
            after_prompt="" if local_answer else lazy_prompt_same_output,
            model=cfg.model,
            before_pricing=cfg.pricing,
            client=client,
            before_output_tokens=BASELINE_OUTPUT_TOKENS,
            after_output_tokens=0 if local_answer else BASELINE_OUTPUT_TOKENS,
            allow_estimate_fallback=not STRICT_GEMINI_COUNTING,
            note="Gemini call skipped because local evidence answers the question." if local_answer else "Fallback to Gemini because local NLP could not answer.",
        ),
        before_prompt=baseline_prompt,
        before_max_output_tokens=BASELINE_OUTPUT_TOKENS,
        prompt="" if local_answer else lazy_prompt_same_output,
        max_output_tokens=0 if local_answer else BASELINE_OUTPUT_TOKENS,
        local_answer_text=local_answer,
    )
)

# 3. Context caching: repeated calls over a stable policy prefix.
cache_prompt = make_agent_prompt(
    question=question,
    context=policy_document,
    history="",
    output_contract=concise_contract,
)
rows.append(
    attach_answer(
        compare_cached_reuse(
            approach="Context caching",
            prompt=cache_prompt,
            static_context=policy_document,
            model=cfg.model,
            pricing=cfg.pricing,
            client=client,
            calls=10,
            output_tokens_per_call=CONCISE_OUTPUT_TOKENS,
            cache_ttl_seconds=cfg.context_cache_ttl_seconds,
            allow_estimate_fallback=not STRICT_GEMINI_COUNTING,
        ),
        before_prompt=cache_prompt,
        before_max_output_tokens=CONCISE_OUTPUT_TOKENS,
        prompt=cache_prompt,
        max_output_tokens=CONCISE_OUTPUT_TOKENS,
        repeat_calls=10,
    )
)

# 4. Context compaction: summarize older turns and keep recent raw turns.
compacted_history = compact_history(sample["history"], keep_last_turns=2, summary_sentences=4)
compacted_prompt = make_agent_prompt(
    question=question,
    context=full_context,
    history=compacted_history,
    output_contract=baseline_contract,
)
rows.append(
    attach_answer(
        compare_prompts(
            approach="Context compaction",
            before_prompt=baseline_prompt,
            after_prompt=compacted_prompt,
            model=cfg.model,
            before_pricing=cfg.pricing,
            client=client,
            before_output_tokens=BASELINE_OUTPUT_TOKENS,
            after_output_tokens=BASELINE_OUTPUT_TOKENS,
            allow_estimate_fallback=not STRICT_GEMINI_COUNTING,
            note="Same context and answer budget; older history is compacted.",
        ),
        before_prompt=baseline_prompt,
        before_max_output_tokens=BASELINE_OUTPUT_TOKENS,
        prompt=compacted_prompt,
        max_output_tokens=BASELINE_OUTPUT_TOKENS,
    )
)

# 5. Output discipline: use retrieved context plus a compact response contract.
verbose_lazy_prompt = make_agent_prompt(
    question=question,
    context=lazy_context,
    history=full_history,
    output_contract=baseline_contract,
)
structured_prompt = make_agent_prompt(
    question=question,
    context=lazy_context,
    history=full_history,
    output_contract=json_contract,
)
rows.append(
    attach_answer(
        compare_prompts(
            approach="Output discipline",
            before_prompt=verbose_lazy_prompt,
            after_prompt=structured_prompt,
            model=cfg.model,
            before_pricing=cfg.pricing,
            client=client,
            before_output_tokens=BASELINE_OUTPUT_TOKENS,
            after_output_tokens=JSON_OUTPUT_TOKENS,
            allow_estimate_fallback=not STRICT_GEMINI_COUNTING,
            note="Input is similar; output contract is much tighter.",
        ),
        before_prompt=verbose_lazy_prompt,
        before_max_output_tokens=BASELINE_OUTPUT_TOKENS,
        prompt=structured_prompt,
        max_output_tokens=JSON_OUTPUT_TOKENS,
    )
)

# 6. Model routing: same concise prompt, cheaper model for simple policy questions.
lite_model = "gemini-2.5-flash-lite"
lite_pricing = DEFAULT_PRICES[lite_model]
concise_lazy_prompt = make_agent_prompt(
    question=question,
    context=lazy_context,
    history=full_history,
    output_contract=concise_contract,
)
rows.append(
    attach_answer(
        compare_prompts(
            approach="Model routing",
            before_prompt=concise_lazy_prompt,
            after_prompt=concise_lazy_prompt,
            model=cfg.model,
            after_model=lite_model,
            before_pricing=cfg.pricing,
            after_pricing=lite_pricing,
            client=client,
            before_output_tokens=CONCISE_OUTPUT_TOKENS,
            after_output_tokens=CONCISE_OUTPUT_TOKENS,
            allow_estimate_fallback=not STRICT_GEMINI_COUNTING,
            note="Token counts are similar; savings come from cheaper model pricing.",
        ),
        before_prompt=concise_lazy_prompt,
        before_max_output_tokens=CONCISE_OUTPUT_TOKENS,
        prompt=concise_lazy_prompt,
        max_output_tokens=CONCISE_OUTPUT_TOKENS,
        model=lite_model,
    )
)

df = pd.DataFrame(rows)
display_columns = [
    "approach",
    "before_model",
    "after_model",
    "count_method",
    "before_input_tokens",
    "after_input_tokens",
    "after_cached_input_tokens",
    "before_output_tokens",
    "after_output_tokens",
    "input_token_saving_pct",
    "output_token_saving_pct",
    "before_cost_usd",
    "after_cost_usd",
    "cost_saving_pct",
    "before_answer",
    "answer_source",
    "optimized_answer",
    "note",
]

df_display = df[display_columns].copy()
for col in ["input_token_saving_pct", "output_token_saving_pct", "cost_saving_pct"]:
    df_display[col] = df_display[col].map(lambda value: f"{value:.1f}%")
for col in ["before_cost_usd", "after_cost_usd"]:
    df_display[col] = df_display[col].map(lambda value: f"${value:.6f}")

df_display


,approach,before_model,after_model,count_method,before_input_tokens,after_input_tokens,after_cached_input_tokens,before_output_tokens,after_output_tokens,input_token_saving_pct,output_token_saving_pct,before_cost_usd,after_cost_usd,cost_saving_pct,before_answer,answer_source,optimized_answer,note
0,Lazy loading context,gemini-2.5-flash,gemini-2.5-flash,gemini,914,523,0,906,773,42.8%,14.7%,$0.002539,$0.002089,17.7%,Here's a detailed customer-facing response and...,live Gemini,Here's a detailed customer-facing response and...,Same answer budget; only irrelevant context is...
1,Local NLP first,gemini-2.5-flash,gemini-2.5-flash,gemini/none,914,0,0,906,0,100.0%,100.0%,$0.002539,$0.000000,100.0%,Here's a detailed customer-facing response and...,local NLP (Gemini skipped by optimization),Eligible: the request is 12 days after purchas...,Gemini call skipped because local evidence ans...
2,Context caching,gemini-2.5-flash,gemini-2.5-flash,gemini,5640,5640,4140,1370,1370,0.0%,0.0%,$0.005117,$0.004261,16.7%,* **Eligibility**: The customer is eligible ...,live Gemini,* **Eligibility**: The customer is eligible ...,10 repeated calls; prompt tokens are unchanged...
3,Context compaction,gemini-2.5-flash,gemini-2.5-flash,gemini,914,874,0,906,693,4.4%,23.5%,$0.002539,$0.001995,21.4%,Here's a detailed customer-facing response and...,live Gemini,Here's a detailed customer-facing response and...,Same context and answer budget; older history ...
4,Output discipline,gemini-2.5-flash,gemini-2.5-flash,gemini,523,522,0,773,182,0.2%,76.5%,$0.002089,$0.000612,70.7%,Here's a detailed customer-facing response and...,live Gemini,"```json\n{\n ""eligible"": true,\n ""customer_r...",Input is similar; output contract is much tigh...
5,Model routing,gemini-2.5-flash,gemini-2.5-flash-lite,gemini,522,522,0,137,161,0.0%,-17.5%,$0.000499,$0.000117,76.6%,* **Eligibility**: The customer is eligible fo...,live Gemini,* **Eligibility:** The customer is eligible ...,Token counts are similar; savings come from ch...


In [7]:
# Show all text in each column
pd.set_option('display.max_colwidth', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

# Prevent line wrapping
pd.set_option('display.expand_frame_repr', False)

# Optional: make display wider
pd.set_option('display.width', 1000)

df_display[["approach","cost_saving_pct","before_answer","answer_source","optimized_answer","note"]]

,approach,cost_saving_pct,before_answer,answer_source,optimized_answer,note
0,Lazy loading context,17.7%,"Here's a detailed customer-facing response and internal support steps for the refund request:\n\n---\n\n**Customer-Facing Response:**\n\nSubject: Your Refund Request for Pro Plan - [Workspace ID]\n\nHi [Customer Name],\n\nThank you for reaching out. I understand you'd like a refund for your annual Pro plan.\n\nI've reviewed your account, and since your purchase was made 12 days ago and your usage is minimal, you are eligible for a full refund according to our self-serve refund policy.\n\nPlease note that once the refund is processed, your Pro plan benefits will be discontinued. The refund typically takes 5-10 business days to appear on your statement, though this can vary depending on your bank.\n\nTo proceed with the refund, please confirm the following details:\n\n* The email address associated with your account.\n* Your Workspace ID.\n* The payment processor ID or invoice number for this purchase.\n\nOnce we have this information, we can process your refund.\n\nBest regards,\n\nThe [Your Company Name] Support Team\n\n---\n\n**Internal Support Steps:**\n\n1. **Verify Customer Information:**\n * Confirm the customer's email address.\n * Confirm the Workspace ID.\n * Confirm the payment processor ID or invoice number.\n * *Policy Reasoning:* This verification is crucial to ensure the refund is applied to the correct account and transaction, as per the ""Before processing, verify the account email, workspace ID, payment processor ID, and invoice number"" policy.\n\n2. **Check Eligibility (Already Confirmed by Conversation History):**\n * **Purchase Date:** 12 days ago (within the 30-day refund window).\n * **Plan Type:** Annual self-serve Pro plan.\n * **Usage:** 200 API calls (below the 10,000 API call abuse threshold).\n * **Region:** Jakarta, Indonesia (follows standard self-serve refund policy).\n * **Refund Type:** Full refund (within the 30-day window, not prorated).\n * *Policy Reasoning:* The ""30-day refund window,"" ""more than 10,000 successful API calls after the purchase"" for blocking refunds, and ""Annual plan refunds should be prorated only after the 30-day window; inside the window, process a full refund"" policies confirm eligibility. The ""Regional notes"" confirm Indonesia follows the standard policy.\n\n3. **Process Refund:**\n * Initiate a **full refund** for the annual Pro plan through the billing system.\n * *Policy Reasoning:* ""inside the window, process a full refund.""\n\n4. **Send Approved Refund Template:**\n * After processing, send the customer the standard approved refund confirmation template.\n * *Policy Reasoning:* ""After processing, send the approved refund template.""\n\n5. **Tag Ticket:**\n * Tag the support ticket with `billing_refund`.\n * *Policy Reasoning:* ""tag the ticket with billing_refund.""\n\n**Missing Evidence:**\n\n* The customer's email address.\n* The customer's Workspace ID.\n* The payment processor ID or invoice number.\n\n**Policy Reasoning Summary:**\n\nThe customer is eligible for a full refund because they are on an annual self-serve plan, requested the refund within the 30-day window (12 days ago), and their API usage (200 calls) is well below the 10,000 API call threshold for refund abuse. Indonesia follows the standard self-serve refund policy. Support can approve this refund as it's a standard self-serve request and not an enterprise contract.\n\n**Regional Caveats:**\n\n* Indonesia follows the standard self-serve refund policy.\n* Local taxes may be non-refundable depending on the payment processor. Support should not provide tax advice; instead, attach the processor receipt and direct the customer to their tax advisor if they inquire about tax specifics.\n\n**Verification Needed:**\n\nBefore processing the refund, the agent must verify the customer's account email, workspace ID, and either the payment processor ID or invoice number to ensure accuracy and pre

## Interpret the Results

The best approach depends on your workload:

- Lazy loading reduces input tokens immediately when the question needs only a few sections.
- Local NLP first can reduce Gemini input and output tokens to zero, but only for deterministic questions with explicit local evidence.
- Caching is best for repeated calls with the same large static prefix. Token count may stay the same, but cached token billing improves the cost.
- Compaction helps long-running chats and agents by replacing older raw turns with summaries.
- Output discipline matters because output tokens are often more expensive than input tokens.
- Model routing reduces cost when a cheaper model is good enough for the task.


In [ ]:
ranked = df.sort_values("cost_saving_pct", ascending=False)[
    ["approach", "cost_saving_pct", "input_token_saving_pct", "output_token_saving_pct", "after_cost_usd"]
].copy()
for col in ["cost_saving_pct", "input_token_saving_pct", "output_token_saving_pct"]:
    ranked[col] = ranked[col].map(lambda value: f"{value:.1f}%")
ranked["after_cost_usd"] = ranked["after_cost_usd"].map(lambda value: f"${value:.6f}")
ranked


## Inspect the Optimized Contexts

These are the actual snippets used by the lazy loading, local NLP, and compaction examples.


In [ ]:
print("--- Lazy-loaded context ---")
print(lazy_context)
print("\n--- Local NLP answer ---")
print(local_answer)
print("\n--- Compacted history ---")
print(compacted_history)


## Optional ADK App Configuration

The cost table uses the Google GenAI SDK for direct token counting because Gemini exposes `count_tokens` there and ADK uses that SDK internally for Gemini models. The helper below shows the equivalent ADK App setup for context caching and context compaction.


In [ ]:
# Optional: requires google-adk and live credentials.
# from config import create_adk_agent, create_adk_app
# root_agent = create_adk_agent(cfg)
# adk_app = create_adk_app(cfg, root_agent)
# adk_app


## Live Usage Metadata

The main comparison table already called Gemini. This cell reuses the stored usage metadata so you can inspect the real `prompt_token_count`, `candidates_token_count`, `cached_content_token_count`, and `total_token_count` values without making duplicate calls.


In [ ]:
usage_df = df[["approach", "before_usage_metadata", "answer_usage_metadata"]].copy()
usage_df
